# CLIP4CAD-GFA v4.9.1 Training (Optimized)

**Direct Contrastive Alignment with Enhanced Text Learning**

## Key Changes from v4.9.0 (which failed with R@1=0%)

### Model Changes
| Parameter | v4.9.0 | v4.9.1 | Rationale |
|-----------|--------|--------|----------|
| d | 256 | 384 | More capacity |
| d_proj | 128 | 256 | Better discrimination |
| d_text_hidden | - | 768 | Gradual text projection |
| num_pool_queries | 8 | 16 | Capture more aspects |
| num_text_tf_layers | 2 | 4 | Deeper text processing |
| num_brep_tf_layers | 4 | 6 | Match text depth |
| Attention pooling layers | 1 | 2 | Better aggregation |

### Training Changes
| Parameter | v4.9.0 | v4.9.1 | Rationale |
|-----------|--------|--------|----------|
| Batch size | 512 | 128 | More gradient updates |
| Stage 0 epochs | 8 | 15 | Stronger BRep-PC anchor |
| Stage 1 epochs | 20 | 40 | More text learning time |
| Text LR multiplier | 1x | 3x | Text needs higher LR |
| Cosine weight | 0.3 | 0.5 | Stronger direct supervision |
| Uniformity loss | 0 | 0.1 | Prevent collapse |

### Loss Changes
- **Asymmetric weighting**: Text losses get 2x weight
- **Direct cosine alignment**: Text should match average geometry
- **Uniformity loss**: Prevent representation collapse
- **Curriculum**: Text weight increases over training

## Expected Results
- Stage 0: BRep-PC margin > 0.4 by epoch 15
- Stage 1: Text-BRep margin > 0.1 by epoch 20, > 0.2 by epoch 40
- Final: R@1 > 10% (vs 0% in v4.9.0)

In [1]:
# Cell 1: Imports and Setup
import sys
sys.path.insert(0, '..')

import os
import gc
import math
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.amp import GradScaler, autocast
from torch.optim import AdamW
from tqdm.auto import tqdm
import numpy as np
from pathlib import Path

# Clean memory
gc.collect()
torch.cuda.empty_cache()

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: NVIDIA GeForce RTX 4090
Memory: 25.8 GB


In [2]:
# Cell 2: Configuration (OPTIMIZED for text learning)
# ═══════════════════════════════════════════════════════════════════════════════
# DATA PATHS
# ═══════════════════════════════════════════════════════════════════════════════
DATA_ROOT = Path("d:/Defect_Det/MMCAD/data")
EMBEDDINGS_DIR = DATA_ROOT / "embeddings"
ALIGNED_DIR = DATA_ROOT / "aligned"

PC_FILE = Path("c:/Users/User/Desktop/pc_embeddings_full.h5")
TEXT_FILE = Path("c:/Users/User/Desktop/text_embeddings.h5")

OUTPUT_DIR = Path("../outputs/gfa_v4_9_1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# TRAINING HYPERPARAMETERS (OPTIMIZED)
# ═══════════════════════════════════════════════════════════════════════════════
BATCH_SIZE = 128  # Smaller batch = more updates (was 512)

# Stage epochs (INCREASED)
STAGE0_EPOCHS = 15    # Anchor BRep to PC (was 8)
STAGE1_EPOCHS = 40    # Full 3-way alignment (was 20)
STAGE2_EPOCHS = 10    # Fine-tuning (was 5)
TOTAL_EPOCHS = STAGE0_EPOCHS + STAGE1_EPOCHS + STAGE2_EPOCHS

# Learning rates
STAGE0_LR = 5e-4      # Higher for faster anchor (was 3e-4)
STAGE1_LR = 2e-4      # Higher for text learning (was 1e-4)
STAGE2_LR = 5e-5      # Fine-tuning (was 2e-5)

# Layer-wise LR multipliers
TEXT_LR_MULT = 3.0    # Text encoder gets 3x LR
POOL_LR_MULT = 2.0    # Attention pooling gets 2x LR

# Warmup epochs per stage
STAGE0_WARMUP = 2
STAGE1_WARMUP = 3
STAGE2_WARMUP = 1

WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

# Evaluation frequency
EVAL_EVERY = 5

print("Configuration (v4.9.1 - Optimized for Text Learning):")
print(f"  Batch size: {BATCH_SIZE} (was 512)")
print(f"  Stage 0: {STAGE0_EPOCHS} epochs, LR={STAGE0_LR}")
print(f"  Stage 1: {STAGE1_EPOCHS} epochs, LR={STAGE1_LR}, Text LR={STAGE1_LR * TEXT_LR_MULT}")
print(f"  Stage 2: {STAGE2_EPOCHS} epochs, LR={STAGE2_LR}")
print(f"  Total: {TOTAL_EPOCHS} epochs")
print(f"  Output: {OUTPUT_DIR}")

Configuration (v4.9.1 - Optimized for Text Learning):
  Batch size: 128 (was 512)
  Stage 0: 15 epochs, LR=0.0005
  Stage 1: 40 epochs, LR=0.0002, Text LR=0.0006000000000000001
  Stage 2: 10 epochs, LR=5e-05
  Total: 65 epochs
  Output: ..\outputs\gfa_v4_9_1


In [3]:
# Cell 3: Load Datasets
from clip4cad.data.gfa_dataset import GFAMappedDataset, gfa_collate_fn

gc.collect()
torch.cuda.empty_cache()

print("Loading datasets...")
print("=" * 60)

MAX_TRAIN_SAMPLES = 111000

# Train dataset
print(f"\n[1/2] Loading TRAIN dataset (max {MAX_TRAIN_SAMPLES:,} samples)...")
train_dataset = GFAMappedDataset(
    data_root=str(DATA_ROOT),
    split="train",
    pc_file=str(PC_FILE),
    text_file=str(TEXT_FILE),
    mapping_dir=str(ALIGNED_DIR),
    load_to_memory=True,
    use_live_text=False,
    use_autobrep=True,
    autobrep_dir=str(EMBEDDINGS_DIR),
    max_samples=MAX_TRAIN_SAMPLES,
)
print(f"Train: {len(train_dataset):,} samples")

# Val dataset
print("\n[2/2] Loading VAL dataset...")
val_dataset = GFAMappedDataset(
    data_root=str(DATA_ROOT),
    split="val",
    pc_file=str(PC_FILE),
    text_file=str(TEXT_FILE),
    mapping_dir=str(ALIGNED_DIR),
    load_to_memory=False,
    use_live_text=False,
    use_autobrep=True,
    autobrep_dir=str(EMBEDDINGS_DIR),
)
print(f"Val: {len(val_dataset):,} samples")

print("\nDATASETS READY!")

Loading datasets...

[1/2] Loading TRAIN dataset (max 111,000 samples)...
  Filtered to 132324 samples with AutoBrep data (from 133105)
  Limiting to 111000 samples (from 132324)
  Loading train data to memory (B-Rep + PC + Text)...
    Loading AutoBrep with topology...
    Loading 111000 samples from train_brep_autobrep.h5...
      Done (111000 samples)
    AutoBrep loaded: 9.2GB in 134.2s
    Loading PC (50GB)...
    Loading Text from: c:\Users\User\Desktop\text_splits\train_text_embeddings.h5
    ✓ Text loaded: 174.6GB in 309.7s
  ✓ Loaded 111000 samples: 205.6GB in RAM (B-Rep + PC + Text)
GFAMappedDataset: train with 111000 samples (in memory)
Train: 111,000 samples

[2/2] Loading VAL dataset...
  Filtered to 16535 samples with AutoBrep data (from 16638)
GFAMappedDataset: val with 16535 samples
Val: 16,535 samples

DATASETS READY!


In [4]:
# Cell 4: Create DataLoaders (smaller batch size)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=gfa_collate_fn,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=gfa_collate_fn,
    pin_memory=True,
)

STEPS_PER_EPOCH = len(train_loader)
print(f"Train loader: {len(train_loader)} batches (4x more than batch_size=512)")
print(f"Val loader: {len(val_loader)} batches")
print(f"Total training steps: {STEPS_PER_EPOCH * TOTAL_EPOCHS:,}")

Train loader: 867 batches (4x more than batch_size=512)
Val loader: 130 batches
Total training steps: 56,355


In [5]:
# Cell 5: Batch Remapping Function
def remap_batch(batch):
    """Remap collate output keys to model expected keys."""
    remapped = {}
    for k, v in batch.items():
        if k.startswith('brep_'):
            new_key = k[5:]
            remapped[new_key] = v
        else:
            remapped[k] = v
    
    if 'pc_features' in remapped:
        pc = remapped.pop('pc_features')
        remapped['pc_local_features'] = pc[:, :-1, :]
        remapped['pc_global_features'] = pc[:, -1, :]
    
    if 'desc_embedding' in remapped:
        remapped['text_features'] = remapped.pop('desc_embedding')
    if 'desc_mask' in remapped:
        remapped['text_mask'] = remapped.pop('desc_mask')
    
    return remapped

# Verify
test_batch = remap_batch(next(iter(train_loader)))
print(f"Batch keys: {list(test_batch.keys())}")
print(f"Text features: {test_batch['text_features'].shape}")

Batch keys: ['has_brep', 'has_pointcloud', 'has_text', 'use_cached_embeddings', 'use_cached_pc_features', 'use_cached_brep_features', 'face_features', 'edge_features', 'face_mask', 'edge_mask', 'edge_to_faces', 'bfs_level', 'face_centroids', 'face_normals', 'face_areas', 'edge_midpoints', 'edge_lengths', 'sample_id', 'idx', 'rot_idx', 'pc_local_features', 'pc_global_features', 'text_features', 'text_mask']
Text features: torch.Size([128, 256, 3072])


In [6]:
# Cell 6: Create Model (ENHANCED configuration)
gc.collect()
torch.cuda.empty_cache()

# Reload module to get latest changes
import importlib
import clip4cad.models.clip4cad_gfa_v4_9 as v49_module
import clip4cad.losses.gfa_v4_9_losses as v49_loss_module
importlib.reload(v49_module)
importlib.reload(v49_loss_module)

from clip4cad.models.clip4cad_gfa_v4_9 import CLIP4CAD_GFA_v49, GFAv49Config
from clip4cad.losses.gfa_v4_9_losses import (
    GFAv49Loss,
    compute_retrieval_metrics,
    compute_contrastive_quality,
    mine_hard_negatives_simple,
    get_warmup_cosine_scheduler,
    get_param_groups_with_lr,
)

# ENHANCED model configuration
config = GFAv49Config(
    # Input dimensions
    d_face=48,
    d_edge=12,
    d_pc=1024,
    d_text=3072,
    # Model dimensions (INCREASED)
    d=384,              # Was 256
    d_proj=256,         # Was 128
    d_text_hidden=768,  # NEW: gradual text projection
    # BRep encoder (DEEPER)
    num_msg_layers=3,
    num_brep_tf_layers=6,   # Was 4
    num_heads=8,
    dropout=0.1,
    # Text encoder (DEEPER)
    num_text_tf_layers=4,   # Was 2
    # Attention pooling (MORE)
    num_pool_queries=16,    # Was 8
    # Temperature
    init_tau=0.07,
)

model = CLIP4CAD_GFA_v49(config).to(device)
model.enable_gradient_checkpointing()

print(f"\nModel: CLIP4CAD_GFA_v49 (ENHANCED)")
print(f"Parameters: {model.count_parameters():,} (was ~12M)")
print(f"\nConfig:")
print(f"  d={config.d}, d_proj={config.d_proj}, d_text_hidden={config.d_text_hidden}")
print(f"  num_pool_queries={config.num_pool_queries}")
print(f"  num_brep_tf_layers={config.num_brep_tf_layers}, num_text_tf_layers={config.num_text_tf_layers}")

c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Enabling gradient checkpointing...
  - BRep transformer: checkpointed
  - Text transformer: checkpointed
Gradient checkpointing enabled

Model: CLIP4CAD_GFA_v49 (ENHANCED)
Parameters: 42,143,620 (was ~12M)

Config:
  d=384, d_proj=256, d_text_hidden=768
  num_pool_queries=16
  num_brep_tf_layers=6, num_text_tf_layers=4


In [7]:
# Cell 7: Loss Function (ENHANCED)
criterion = GFAv49Loss(
    label_smoothing=0.1,      # Was 0.05
    cosine_weight=0.5,        # Was 0.3
    uniformity_weight=0.1,    # NEW: prevent collapse
    text_weight=2.0,          # NEW: upweight text losses
)

scaler = GradScaler('cuda')

print("Loss: GFAv49Loss (ENHANCED)")
print(f"  label_smoothing={criterion.label_smoothing}")
print(f"  cosine_weight={criterion.cosine_weight}")
print(f"  uniformity_weight={criterion.uniformity_weight}")
print(f"  text_weight={criterion.text_weight}")

Loss: GFAv49Loss (ENHANCED)
  label_smoothing=0.1
  cosine_weight=0.5
  uniformity_weight=0.1
  text_weight=2.0


In [13]:
# === REIMPORT v4.9.2 CHANGES ===
import importlib
import clip4cad.losses.gfa_v4_9_losses as v49_loss_module
importlib.reload(v49_loss_module)

from clip4cad.losses.gfa_v4_9_losses import (
    GFAv49Loss,
    compute_retrieval_metrics,
    compute_contrastive_quality,
    compute_batch_r1,        # NEW
    diagnose_embeddings,     # NEW
    mine_hard_negatives_simple,
    get_warmup_cosine_scheduler,
    get_param_groups_with_lr,
)

# NEW loss with v4.9.2 settings (NO cosine alignment!)
criterion = GFAv49Loss(
    label_smoothing=0.01,      # 10x lower (was 0.1)
    cosine_weight=0.0,          # REMOVED (was 0.5)
    uniformity_weight=0.5,      # 5x higher (was 0.1)
    text_weight=1.5,
    variance_weight=0.1,        # NEW
)

print("v4.9.2 Loss reloaded:")
print(f"  label_smoothing={criterion.label_smoothing}")
print(f"  cosine_weight={criterion.cosine_weight} (IGNORED)")
print(f"  uniformity_weight={criterion.uniformity_weight}")
print(f"  variance_weight={criterion.variance_weight}")

v4.9.2 Loss reloaded:
  label_smoothing=0.01
  cosine_weight=0.0 (IGNORED)
  uniformity_weight=0.5
  variance_weight=0.1


In [8]:
# Cell 8: Verify Forward Pass
print("Verifying forward pass...")

model.eval()
with torch.no_grad():
    test_batch = remap_batch(next(iter(train_loader)))
    with autocast('cuda'):
        # Stage 0
        outputs_s0 = model.forward_stage0(test_batch)
        loss_s0, loss_dict_s0 = criterion(outputs_s0, stage=0, epoch=1, total_epochs=STAGE0_EPOCHS)
        
        # Stage 1
        outputs_s1 = model(test_batch, stage=1)
        loss_s1, loss_dict_s1 = criterion(outputs_s1, stage=1, epoch=1, total_epochs=STAGE1_EPOCHS)

print(f"\nStage 0: loss={loss_s0.item():.4f}, margin={loss_dict_s0['margin'].item():.4f}")
print(f"Stage 1: loss={loss_s1.item():.4f}, margin={loss_dict_s1['margin'].item():.4f}")
print(f"  margin_tb={loss_dict_s1['margin_tb'].item():.4f}")
print(f"  margin_tp={loss_dict_s1['margin_tp'].item():.4f}")
print(f"  margin_bp={loss_dict_s1['margin_bp'].item():.4f}")
print("\nForward pass OK!")

Verifying forward pass...

Stage 0: loss=5.5705, margin=-0.0009
Stage 1: loss=5.8916, margin=-0.0004
  margin_tb=-0.0003
  margin_tp=0.0001
  margin_bp=-0.0009

Forward pass OK!


## Stage 0: Anchor BRep to PC (15 epochs)

**Goal:** Strong BRep-PC alignment before adding text.

**Expected:** margin > 0.4 by epoch 15

In [9]:
# Cell 9: Stage 0 Training
print("\n" + "="*70)
print("STAGE 0: Anchor BRep to PC (15 epochs)")
print("="*70)

model.freeze_pc_encoder()

# Optimizer
optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=STAGE0_LR,
    weight_decay=WEIGHT_DECAY
)
scheduler = get_warmup_cosine_scheduler(
    optimizer, STAGE0_WARMUP, STAGE0_EPOCHS, STEPS_PER_EPOCH
)

stage0_history = {'loss': [], 'margin': [], 'pos_sim': [], 'neg_sim': []}
global_epoch = 0

for epoch in range(1, STAGE0_EPOCHS + 1):
    global_epoch += 1
    model.train()
    epoch_metrics = {'loss': [], 'margin': [], 'pos_sim': [], 'neg_sim': []}
    
    pbar = tqdm(train_loader, desc=f"S0 Ep {epoch}/{STAGE0_EPOCHS}")
    for batch in pbar:
        batch = remap_batch(batch)
        
        with autocast('cuda'):
            outputs = model.forward_stage0(batch)
            loss, loss_dict = criterion(outputs, stage=0, epoch=epoch, total_epochs=STAGE0_EPOCHS)
        
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        epoch_metrics['loss'].append(loss_dict['total'].item())
        epoch_metrics['margin'].append(loss_dict['margin'].item())
        epoch_metrics['pos_sim'].append(loss_dict['pos_sim'].item())
        epoch_metrics['neg_sim'].append(loss_dict['neg_sim'].item())
        
        pbar.set_postfix({'L': f"{loss.item():.3f}", 'M': f"{loss_dict['margin'].item():.3f}"})
    
    # Log
    for k in epoch_metrics:
        stage0_history[k].append(np.mean(epoch_metrics[k]))
    
    print(f"Ep {epoch}: Loss={stage0_history['loss'][-1]:.4f}, "
          f"Margin={stage0_history['margin'][-1]:.4f}, "
          f"Pos={stage0_history['pos_sim'][-1]:.3f}, Neg={stage0_history['neg_sim'][-1]:.3f}")

# Save
torch.save({
    'epoch': STAGE0_EPOCHS,
    'model_state_dict': model.state_dict(),
    'history': stage0_history,
    'config': config,
}, OUTPUT_DIR / 'checkpoint_stage0.pt')

print(f"\nStage 0 done! Margin: {stage0_history['margin'][-1]:.4f}")


STAGE 0: Anchor BRep to PC (15 epochs)
PC encoder frozen (anchor mode)


S0 Ep 1/15:   0%|          | 0/867 [00:00<?, ?it/s]

c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:192: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


Ep 1: Loss=3.8500, Margin=0.0993, Pos=0.112, Neg=0.013


S0 Ep 2/15:   0%|          | 0/867 [00:00<?, ?it/s]

...

Ep 13: Loss=0.7593, Margin=0.1419, Pos=0.160, Neg=0.018


S0 Ep 14/15:   0%|          | 0/867 [00:00<?, ?it/s]

Ep 14: Loss=0.7470, Margin=0.1414, Pos=0.159, Neg=0.017


S0 Ep 15/15:   0%|          | 0/867 [00:00<?, ?it/s]

Ep 15: Loss=0.7396, Margin=0.1412, Pos=0.158, Neg=0.017

Stage 0 done! Margin: 0.1412


In [10]:
# Cell 9b: Load Stage 0 (alternative)
CKPT = OUTPUT_DIR / 'checkpoint_stage0.pt'
if CKPT.exists():
    ckpt = torch.load(CKPT, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    stage0_history = ckpt['history']
    global_epoch = STAGE0_EPOCHS
    print(f"Loaded Stage 0: Margin={stage0_history['margin'][-1]:.4f}")
else:
    print("No Stage 0 checkpoint, run Cell 9 first")

Loaded Stage 0: Margin=0.1412


## Stage 1: Full 3-Way Alignment (40 epochs)

**Goal:** Align text with geometry using:
- Layer-wise LR (text gets 3x)
- Asymmetric loss weighting (text losses get 2x)
- Direct cosine supervision
- Uniformity regularization

**Expected:**
- Epoch 10: margin_tb > 0.05
- Epoch 20: margin_tb > 0.1, R@1 > 2%
- Epoch 40: margin_tb > 0.2, R@1 > 10%

In [11]:
# Cell 10: Stage 1 Training
print("\n" + "="*70)
print("STAGE 1: Full 3-Way Alignment (40 epochs)")
print("="*70)

model.unfreeze_pc_encoder()

# Layer-wise LR
param_groups = get_param_groups_with_lr(
    model,
    base_lr=STAGE1_LR,
    text_lr_mult=TEXT_LR_MULT,
    pool_lr_mult=POOL_LR_MULT,
    weight_decay=WEIGHT_DECAY,
)
optimizer = AdamW(param_groups)
scheduler = get_warmup_cosine_scheduler(
    optimizer, STAGE1_WARMUP, STAGE1_EPOCHS, STEPS_PER_EPOCH
)

stage1_history = {
    'loss': [], 'margin': [], 'margin_tb': [], 'margin_tp': [], 'margin_bp': [],
    'infonce': [], 'cosine_tg': [], 'uniformity': []
}
eval_history = {'epoch': [], 'r1': [], 'r5': [], 'r10': []}

for epoch in range(1, STAGE1_EPOCHS + 1):
    global_epoch += 1
    model.train()
    epoch_metrics = {k: [] for k in stage1_history.keys()}
    
    pbar = tqdm(train_loader, desc=f"S1 Ep {epoch}/{STAGE1_EPOCHS}")
    for batch in pbar:
        batch = remap_batch(batch)
        
        with autocast('cuda'):
            outputs = model(batch, stage=1)
            loss, loss_dict = criterion(
                outputs, stage=1, epoch=epoch, total_epochs=STAGE1_EPOCHS
            )
        
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        epoch_metrics['loss'].append(loss_dict['total'].item())
        epoch_metrics['margin'].append(loss_dict['margin'].item())
        epoch_metrics['margin_tb'].append(loss_dict['margin_tb'].item())
        epoch_metrics['margin_tp'].append(loss_dict['margin_tp'].item())
        epoch_metrics['margin_bp'].append(loss_dict['margin_bp'].item())
        epoch_metrics['infonce'].append(loss_dict['infonce'].item())
        epoch_metrics['cosine_tg'].append(loss_dict.get('cosine_tg', torch.tensor(0)).item())
        epoch_metrics['uniformity'].append(
            loss_dict.get('uniformity_text', torch.tensor(0)).item()
        )
        
        pbar.set_postfix({
            'L': f"{loss.item():.2f}",
            'TB': f"{loss_dict['margin_tb'].item():.3f}",
        })
    
    # Log
    for k in epoch_metrics:
        stage1_history[k].append(np.mean(epoch_metrics[k]))
    
    print(f"Ep {epoch}: L={stage1_history['loss'][-1]:.3f}, "
          f"TB={stage1_history['margin_tb'][-1]:.4f}, "
          f"TP={stage1_history['margin_tp'][-1]:.4f}, "
          f"BP={stage1_history['margin_bp'][-1]:.4f}")
    
    # Evaluate
    if epoch % EVAL_EVERY == 0 or epoch == STAGE1_EPOCHS:
        model.eval()
        all_z_text, all_z_brep = [], []
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="  Eval", leave=False):
                batch = remap_batch(batch)
                with autocast('cuda'):
                    out = model(batch, stage=1)
                all_z_text.append(out['z_text'].cpu())
                all_z_brep.append(out['z_brep'].cpu())
        
        z_text = torch.cat(all_z_text, dim=0)
        z_brep = torch.cat(all_z_brep, dim=0)
        metrics = compute_retrieval_metrics(z_text, z_brep)
        
        eval_history['epoch'].append(epoch)
        eval_history['r1'].append(metrics['R@1'])
        eval_history['r5'].append(metrics['R@5'])
        eval_history['r10'].append(metrics['R@10'])
        
        # Also compute quality
        quality = compute_contrastive_quality(z_text, z_brep)
        print(f"  -> R@1={metrics['R@1']:.1f}%, R@5={metrics['R@5']:.1f}%, "
              f"eval_margin={quality['margin']:.4f}")
        model.train()

# Save
torch.save({
    'epoch': STAGE0_EPOCHS + STAGE1_EPOCHS,
    'model_state_dict': model.state_dict(),
    'stage0_history': stage0_history,
    'stage1_history': stage1_history,
    'eval_history': eval_history,
    'config': config,
}, OUTPUT_DIR / 'checkpoint_stage1.pt')

print(f"\nStage 1 done! TB margin: {stage1_history['margin_tb'][-1]:.4f}")
if eval_history['r1']:
    print(f"Final R@1: {eval_history['r1'][-1]:.1f}%")


STAGE 1: Full 3-Way Alignment (40 epochs)
PC encoder unfrozen
Parameter groups:
  BRep/PC encoders: 150 params, LR=0.0002
  Text encoder: 95 params, LR=0.0006000000000000001
  Attention pools: 66 params, LR=0.0004


S1 Ep 1/40:   0%|          | 0/867 [00:00<?, ?it/s]

Ep 1: L=2.899, TB=0.0997, TP=-0.0000, BP=0.1364


S1 Ep 2/40:   0%|          | 0/867 [00:00<?, ?it/s]

Ep 2: L=1.936, TB=0.1824, TP=0.0000, BP=0.1396


...

Ep 39: L=0.199, TB=0.4232, TP=0.4187, BP=0.4143


S1 Ep 40/40:   0%|          | 0/867 [00:00<?, ?it/s]

Ep 40: L=0.198, TB=0.4230, TP=0.4188, BP=0.4148


  Eval:   0%|          | 0/130 [00:00<?, ?it/s]

  -> R@1=0.0%, R@5=0.0%, eval_margin=0.0000

Stage 1 done! TB margin: 0.4230
Final R@1: 0.0%


## Stage 2: Fine-Tuning with Hard Negatives (10 epochs)

In [12]:
# Cell 11: Stage 2 Training
print("\n" + "="*70)
print("STAGE 2: Fine-Tuning with Hard Negatives (10 epochs)")
print("="*70)

# Mine hard negatives
hard_negatives = mine_hard_negatives_simple(
    model, train_loader, device,
    top_k=10, max_batches=100,
    remap_fn=remap_batch
)

# Optimizer with lower LR
param_groups = get_param_groups_with_lr(
    model, base_lr=STAGE2_LR, text_lr_mult=TEXT_LR_MULT,
    pool_lr_mult=POOL_LR_MULT, weight_decay=WEIGHT_DECAY
)
optimizer = AdamW(param_groups)
scheduler = get_warmup_cosine_scheduler(
    optimizer, STAGE2_WARMUP, STAGE2_EPOCHS, STEPS_PER_EPOCH
)

stage2_history = {'loss': [], 'margin': [], 'margin_tb': []}
best_r1 = eval_history['r1'][-1] if eval_history['r1'] else 0

for epoch in range(1, STAGE2_EPOCHS + 1):
    global_epoch += 1
    
    # Re-mine every 3 epochs
    if epoch > 1 and epoch % 3 == 0:
        print("Re-mining hard negatives...")
        hard_negatives = mine_hard_negatives_simple(
            model, train_loader, device,
            top_k=10, max_batches=100,
            remap_fn=remap_batch
        )
    
    model.train()
    epoch_metrics = {'loss': [], 'margin': [], 'margin_tb': []}
    
    pbar = tqdm(train_loader, desc=f"S2 Ep {epoch}/{STAGE2_EPOCHS}")
    for batch_idx, batch in enumerate(pbar):
        batch = remap_batch(batch)
        batch_size = batch['face_features'].shape[0]
        start_idx = batch_idx * batch_size
        batch_hard_negs = [
            hard_negatives.get(start_idx + i) for i in range(batch_size)
        ]
        
        with autocast('cuda'):
            outputs = model(batch, stage=1)
            loss, loss_dict = criterion(
                outputs, stage=2,
                hard_negatives=batch_hard_negs,
                hard_neg_boost=1.5
            )
        
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        epoch_metrics['loss'].append(loss_dict['total'].item())
        epoch_metrics['margin'].append(loss_dict['margin'].item())
        epoch_metrics['margin_tb'].append(loss_dict['margin_tb'].item())
        
        pbar.set_postfix({'L': f"{loss.item():.2f}", 'TB': f"{loss_dict['margin_tb'].item():.3f}"})
    
    for k in epoch_metrics:
        stage2_history[k].append(np.mean(epoch_metrics[k]))
    
    print(f"Ep {epoch}: L={stage2_history['loss'][-1]:.3f}, TB={stage2_history['margin_tb'][-1]:.4f}")

# Final eval
print("\nFinal evaluation...")
model.eval()
all_z_text, all_z_brep, all_z_pc = [], [], []
with torch.no_grad():
    for batch in tqdm(val_loader, desc="Encoding"):
        batch = remap_batch(batch)
        with autocast('cuda'):
            out = model(batch, stage=1)
        all_z_text.append(out['z_text'].cpu())
        all_z_brep.append(out['z_brep'].cpu())
        all_z_pc.append(out['z_pc'].cpu())

z_text = torch.cat(all_z_text, dim=0)
z_brep = torch.cat(all_z_brep, dim=0)
z_pc = torch.cat(all_z_pc, dim=0)

metrics_tb = compute_retrieval_metrics(z_text, z_brep)
metrics_tp = compute_retrieval_metrics(z_text, z_pc)
metrics_bp = compute_retrieval_metrics(z_brep, z_pc)

quality_tb = compute_contrastive_quality(z_text, z_brep)
quality_tp = compute_contrastive_quality(z_text, z_pc)
quality_bp = compute_contrastive_quality(z_brep, z_pc)

print("\n" + "="*60)
print("FINAL RESULTS:")
print("="*60)
print(f"Text→BRep: R@1={metrics_tb['R@1']:.1f}%, R@5={metrics_tb['R@5']:.1f}%, margin={quality_tb['margin']:.4f}")
print(f"Text→PC:   R@1={metrics_tp['R@1']:.1f}%, R@5={metrics_tp['R@5']:.1f}%, margin={quality_tp['margin']:.4f}")
print(f"BRep→PC:   R@1={metrics_bp['R@1']:.1f}%, R@5={metrics_bp['R@5']:.1f}%, margin={quality_bp['margin']:.4f}")
print("="*60)

# Save
torch.save({
    'epoch': TOTAL_EPOCHS,
    'model_state_dict': model.state_dict(),
    'stage0_history': stage0_history,
    'stage1_history': stage1_history,
    'stage2_history': stage2_history,
    'eval_history': eval_history,
    'final_metrics': {
        'tb': metrics_tb, 'tp': metrics_tp, 'bp': metrics_bp,
        'quality_tb': quality_tb, 'quality_tp': quality_tp, 'quality_bp': quality_bp
    },
    'config': config,
}, OUTPUT_DIR / 'gfa_v4_9_1_final.pt')

print(f"\nModel saved to {OUTPUT_DIR / 'gfa_v4_9_1_final.pt'}")


STAGE 2: Fine-Tuning with Hard Negatives (10 epochs)
Mining hard negatives by embedding similarity...
Mined hard negatives for 12800 samples
Parameter groups:
  BRep/PC encoders: 150 params, LR=5e-05
  Text encoder: 95 params, LR=0.00015000000000000001
  Attention pools: 66 params, LR=0.0001


S2 Ep 1/10:   0%|          | 0/867 [00:00<?, ?it/s]

Ep 1: L=0.492, TB=0.3960


S2 Ep 2/10:   0%|          | 0/867 [00:00<?, ?it/s]

Ep 2: L=0.491, TB=0.3882
Re-mining hard negatives...
Mining hard negatives by embedding similarity...
Mined hard negatives for 12800 samples


...

Ep 6: L=0.487, TB=0.3821


S2 Ep 7/10:   0%|          | 0/867 [00:00<?, ?it/s]

Ep 7: L=0.485, TB=0.3815


S2 Ep 8/10:   0%|          | 0/867 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Cell 12: Visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Stage 0: Margin
ax = axes[0, 0]
ax.plot(stage0_history['margin'], 'b-', linewidth=2)
ax.axhline(y=0.4, color='g', linestyle='--', alpha=0.5, label='Target')
ax.set_xlabel('Epoch')
ax.set_ylabel('Margin')
ax.set_title('Stage 0: BRep-PC Margin')
ax.legend()
ax.grid(True, alpha=0.3)

# Stage 1: All margins
ax = axes[0, 1]
ax.plot(stage1_history['margin_tb'], 'r-', label='Text-BRep', linewidth=2)
ax.plot(stage1_history['margin_tp'], 'g-', label='Text-PC', linewidth=2)
ax.plot(stage1_history['margin_bp'], 'b-', label='BRep-PC', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Margin')
ax.set_title('Stage 1: 3-Way Margins')
ax.legend()
ax.grid(True, alpha=0.3)

# Retrieval
ax = axes[0, 2]
if eval_history['epoch']:
    ax.plot(eval_history['epoch'], eval_history['r1'], 'ro-', label='R@1', linewidth=2)
    ax.plot(eval_history['epoch'], eval_history['r5'], 'gs-', label='R@5', linewidth=2)
    ax.plot(eval_history['epoch'], eval_history['r10'], 'b^-', label='R@10', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Recall (%)')
ax.set_title('Text→BRep Retrieval')
ax.legend()
ax.grid(True, alpha=0.3)

# Loss
ax = axes[1, 0]
all_loss = stage0_history['loss'] + stage1_history['loss'] + stage2_history['loss']
ax.plot(all_loss, 'k-', linewidth=2)
ax.axvline(x=STAGE0_EPOCHS, color='r', linestyle='--', alpha=0.5)
ax.axvline(x=STAGE0_EPOCHS + STAGE1_EPOCHS, color='r', linestyle='--', alpha=0.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.grid(True, alpha=0.3)

# Cosine alignment
ax = axes[1, 1]
if 'cosine_tg' in stage1_history:
    ax.plot(stage1_history['cosine_tg'], 'r-', label='Cosine T-G', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Cosine Loss')
ax.set_title('Stage 1: Text-Geometry Cosine')
ax.legend()
ax.grid(True, alpha=0.3)

# Summary
axes[1, 2].axis('off')
summary = f"""GFA v4.9.1 Training Summary
(Optimized for Text Learning)

Model: {model.count_parameters():,} params
d={config.d}, d_proj={config.d_proj}

Stage 0: {STAGE0_EPOCHS} epochs
  BP margin: {stage0_history['margin'][-1]:.4f}

Stage 1: {STAGE1_EPOCHS} epochs
  TB margin: {stage1_history['margin_tb'][-1]:.4f}
  R@1: {eval_history['r1'][-1] if eval_history['r1'] else 0:.1f}%

Stage 2: {STAGE2_EPOCHS} epochs
  TB margin: {stage2_history['margin_tb'][-1]:.4f}

Final Text→BRep R@1: {metrics_tb['R@1']:.1f}%
Final eval margin: {quality_tb['margin']:.4f}"""
axes[1, 2].text(0.5, 0.5, summary, ha='center', va='center',
                fontsize=10, family='monospace',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150)
plt.show()

In [14]:
# === DIAGNOSE CURRENT MODEL STATE ===
print("Diagnosing current embedding state...")

model.eval()
all_z_text, all_z_brep, all_z_pc = [], [], []

with torch.no_grad():
    for i, batch in enumerate(val_loader):
        if i >= 20:  # Sample 20 batches
            break
        batch = remap_batch(batch)
        with autocast('cuda'):
            out = model(batch, stage=1)
        all_z_text.append(out['z_text'].cpu())
        all_z_brep.append(out['z_brep'].cpu())
        all_z_pc.append(out['z_pc'].cpu())

z_text = torch.cat(all_z_text, dim=0)
z_brep = torch.cat(all_z_brep, dim=0)
z_pc = torch.cat(all_z_pc, dim=0)

diag = diagnose_embeddings(z_text, z_brep, z_pc)

print("\n" + "="*60)
print("DIAGNOSIS (v4.9.2)")
print("="*60)
print(f"\nVariance (should be > 0.3):")
print(f"  Text:  {diag['var_text']:.4f}")
print(f"  BRep:  {diag['var_brep']:.4f}")
print(f"  PC:    {diag['var_pc']:.4f}")

print(f"\nBatch R@1 (random chance = {diag['random_chance']:.1f}%):")
print(f"  T→B: {diag['r1_tb']:.1f}%")
print(f"  T→P: {diag['r1_tp']:.1f}%")
print(f"  B→P: {diag['r1_bp']:.1f}%")

print(f"\nMargins:")
print(f"  T-B: {diag['margin_tb']:.4f} (pos={diag['pos_tb']:.3f}, neg={diag['neg_tb']:.3f})")
print(f"  T-P: {diag['margin_tp']:.4f} (pos={diag['pos_tp']:.3f}, neg={diag['neg_tp']:.3f})")
print(f"  B-P: {diag['margin_bp']:.4f} (pos={diag['pos_bp']:.3f}, neg={diag['neg_bp']:.3f})")

# Full eval
metrics = compute_retrieval_metrics(z_text, z_brep)
print(f"\nFull R@1: {metrics['R@1']:.1f}%")
print("="*60)



Diagnosing current embedding state...

DIAGNOSIS (v4.9.2)

Variance (should be > 0.3):
  Text:  0.0000
  BRep:  0.0000
  PC:    0.0000

Batch R@1 (random chance = 0.0%):
  T→B: 0.1%
  T→P: 0.0%
  B→P: 0.0%

Margins:
  T-B: 0.0000 (pos=-0.444, neg=-0.444)
  T-P: 0.0000 (pos=0.192, neg=0.192)
  B-P: 0.0000 (pos=-0.292, neg=-0.292)

Full R@1: 0.0%


In [16]:
# === CLEAR GPU AND RELOAD ===                                                                                                                                                                                                               import gc
# Delete model and optimizer if they exist
if 'model' in dir():
    del model
if 'optimizer' in dir():
    del optimizer
if 'scheduler' in dir():
    del scheduler
if 'scaler' in dir():
    del scaler

gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after clear: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# Reload modules with v4.9.2 changes
import importlib
import clip4cad.losses.gfa_v4_9_losses as v49_loss_module
import clip4cad.models.clip4cad_gfa_v4_9 as v49_module
importlib.reload(v49_loss_module)
importlib.reload(v49_module)

from clip4cad.models.clip4cad_gfa_v4_9 import CLIP4CAD_GFA_v49, GFAv49Config
from clip4cad.losses.gfa_v4_9_losses import (
    GFAv49Loss,
    compute_retrieval_metrics,
    compute_contrastive_quality,
    compute_batch_r1,
    diagnose_embeddings,
    mine_hard_negatives_simple,
    get_warmup_cosine_scheduler,
    get_param_groups_with_lr,
)

# Recreate model
config = GFAv49Config(
    d_face=48, d_edge=12, d_pc=1024, d_text=3072,
    d=384, d_proj=256, d_text_hidden=768,
    num_msg_layers=3, num_brep_tf_layers=6, num_heads=8, dropout=0.1,
    num_text_tf_layers=4, num_pool_queries=16, init_tau=0.07,
)

model = CLIP4CAD_GFA_v49(config).to(device)
model.enable_gradient_checkpointing()

# Load Stage 1 checkpoint
ckpt = torch.load(OUTPUT_DIR / 'checkpoint_stage1.pt', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded Stage 1 checkpoint")

# Create v4.9.2 loss (NO cosine alignment!)
criterion = GFAv49Loss(
    label_smoothing=0.01,
    cosine_weight=0.0,
    uniformity_weight=0.5,
    text_weight=1.5,
    variance_weight=0.1,
)

scaler = GradScaler('cuda')

print(f"\nModel loaded: {model.count_parameters():,} params")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"\nv4.9.2 Loss: label_smooth={criterion.label_smoothing}, "
    f"uniform_w={criterion.uniformity_weight}, var_w={criterion.variance_weight}")

GPU memory after clear: 0.55 GB


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Enabling gradient checkpointing...
  - BRep transformer: checkpointed
  - Text transformer: checkpointed
Gradient checkpointing enabled
Loaded Stage 1 checkpoint

Model loaded: 42,143,620 params
GPU memory: 0.72 GB

v4.9.2 Loss: label_smooth=0.01, uniform_w=0.5, var_w=0.1


In [17]:





param_groups = get_param_groups_with_lr(
    model,
    base_lr=1e-4,  # Slightly lower
    text_lr_mult=3.0,
    pool_lr_mult=2.0,
    weight_decay=0.01,
)
optimizer = AdamW(param_groups)
scheduler = get_warmup_cosine_scheduler(
    optimizer, 2, STAGE1_V2_EPOCHS, STEPS_PER_EPOCH
)

print(f"\nStage 1 v4.9.2: {STAGE1_V2_EPOCHS} more epochs with PURE InfoNCE")

for epoch in range(1, STAGE1_V2_EPOCHS + 1):
    model.train()
    epoch_metrics = {'loss': [], 'margin_tb': [], 'r1_batch': [], 'var_text': []}

    pbar = tqdm(train_loader, desc=f"S1v2 Ep {epoch}/{STAGE1_V2_EPOCHS}")
    for batch in pbar:
        batch = remap_batch(batch)

        with autocast('cuda'):
            outputs = model(batch, stage=1)
            loss, loss_dict = criterion(
                outputs, stage=1, epoch=epoch, total_epochs=STAGE1_V2_EPOCHS
            )

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        # Track batch R@1 (CRITICAL for detecting collapse)
        with torch.no_grad():
            batch_r1 = compute_batch_r1(outputs['z_text'], outputs['z_brep'])

        epoch_metrics['loss'].append(loss_dict['total'].item())
        epoch_metrics['margin_tb'].append(loss_dict['margin_tb'].item())
        epoch_metrics['r1_batch'].append(batch_r1)
        epoch_metrics['var_text'].append(loss_dict.get('z_text_var', torch.tensor(0)).item())

        pbar.set_postfix({
            'L': f"{loss.item():.2f}",
            'TB': f"{loss_dict['margin_tb'].item():.3f}",
            'R1b': f"{batch_r1:.1f}%",
        })

    # Log
    avg_loss = np.mean(epoch_metrics['loss'])
    avg_margin = np.mean(epoch_metrics['margin_tb'])
    avg_r1 = np.mean(epoch_metrics['r1_batch'])
    avg_var = np.mean(epoch_metrics['var_text'])

    print(f"Ep {epoch}: L={avg_loss:.3f}, TB={avg_margin:.4f}, "
        f"batch_R1={avg_r1:.1f}%, var_text={avg_var:.4f}")

    # Evaluate every 5 epochs
    if epoch % 5 == 0:
        model.eval()
        all_z_text, all_z_brep = [], []
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="  Eval", leave=False):
                batch = remap_batch(batch)
                with autocast('cuda'):
                    out = model(batch, stage=1)
                all_z_text.append(out['z_text'].cpu())
                all_z_brep.append(out['z_brep'].cpu())

        z_text = torch.cat(all_z_text, dim=0)
        z_brep = torch.cat(all_z_brep, dim=0)
        metrics = compute_retrieval_metrics(z_text, z_brep)
        quality = compute_contrastive_quality(z_text, z_brep)

        print(f"  -> R@1={metrics['R@1']:.1f}%, R@5={metrics['R@5']:.1f}%, "
            f"eval_margin={quality['margin']:.4f}")
        model.train()

# Save
torch.save({
    'model_state_dict': model.state_dict(),
    'config': config,
}, OUTPUT_DIR / 'checkpoint_v4_9_2.pt')
print(f"\nSaved to {OUTPUT_DIR / 'checkpoint_v4_9_2.pt'}")

Parameter groups:
  BRep/PC encoders: 150 params, LR=0.0001
  Text encoder: 95 params, LR=0.00030000000000000003
  Attention pools: 66 params, LR=0.0002

Stage 1 v4.9.2: 20 more epochs with PURE InfoNCE


S1v2 Ep 1/20:   0%|          | 0/867 [00:00<?, ?it/s]

Ep 1: L=-1.494, TB=0.4300, batch_R1=99.1%, var_text=0.0039


S1v2 Ep 2/20:   0%|          | 0/867 [00:00<?, ?it/s]

Ep 2: L=-1.564, TB=0.4649, batch_R1=100.0%, var_text=0.0039


...

Ep 19: L=-1.621, TB=0.4986, batch_R1=100.0%, var_text=0.0039


S1v2 Ep 20/20:   0%|          | 0/867 [00:00<?, ?it/s]

Ep 20: L=-1.621, TB=0.4990, batch_R1=100.0%, var_text=0.0039


  Eval:   0%|          | 0/130 [00:00<?, ?it/s]

  -> R@1=0.0%, R@5=0.0%, eval_margin=0.0000

Saved to ..\outputs\gfa_v4_9_1\checkpoint_v4_9_2.pt
